In [0]:
# service_credential = dbutils.secrets.get(scope='scope-test-learning-awg', key='azure-ms-entra-secret-key-vault-governance-awg')
 
# spark.conf.set("fs.azure.account.auth.type.datalaketestlearningawg.dfs.core.windows.net", "OAuth")
# spark.conf.set("fs.azure.account.oauth.provider.type.datalaketestlearningawg.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
# spark.conf.set("fs.azure.account.oauth2.client.id.datalaketestlearningawg.dfs.core.windows.net", "504d82b4-c2c8-496f-9de9-b944b4f1b9f6")
# spark.conf.set("fs.azure.account.oauth2.client.secret.datalaketestlearningawg.dfs.core.windows.net", service_credential)
# spark.conf.set("fs.azure.account.oauth2.client.endpoint.datalaketestlearningawg.dfs.core.windows.net", "https://login.microsoftonline.com/50a3af05-f42e-4ad1-9203-9b4d0f7b4664/oauth2/token")

In [0]:
# from pyspark.sql import Row
# import json

# data = [{"msg": "test"}, {"msg": "test2"}]

# # Create DataFrame directly from list of Rows
# df = spark.createDataFrame([Row(value=json.dumps(d)) for d in data])

# # Write as text to ADLS
# df.write.mode("overwrite").text(
#     "abfss://samco-streaming-container@datalaketestlearningawg.dfs.core.windows.net/source/test.json"
# )

In [0]:
# with open("/Volumes/main/default/vol1/test222/test.json", "w") as f:
#     f.write("[{'msg': 'test'}, {'msg': 'test2'}]" + "\n")

In [0]:
# import os
# # Create the directory if it doesn't exist
# os.makedirs("/Volumes/main/default/vol2/2885_NSE/XXX", exist_ok=True)

In [0]:
import os
import json
import websocket
import threading
import time
from queue import Queue
from snapi_py_client.snapi_bridge import StocknoteAPIPythonBridge

userId = dbutils.secrets.get(scope='scope-test-learning-awg', key='samco-username')
password = dbutils.secrets.get(scope='scope-test-learning-awg', key='samco-password')
yob = dbutils.secrets.get(scope='scope-test-learning-awg', key='samco-yob')
output_path = "abfss://samco-streaming-container@datalaketestlearningawg.dfs.core.windows.net/source"
output_path = "/Volumes/main/default/vol2/2497_MCX"
output_path = "/Volumes/main/default/vol2/2885_NSE"
symbol = "2885_NSE"


class SamcoMarketStreamer:
    def __init__(self, userId, password, yob, output_path):
        self.userId = userId
        self.password = password
        self.yob = yob
        self.output_path = output_path

        self.samco = StocknoteAPIPythonBridge()
        self.session_token = None

        self.ws = None
        self.tick_queue = Queue()
        self.stop_event = threading.Event()

        self.symbol = None

        self.login()

    # ---------- LOGIN ----------

    def login(self):

        login = self.samco.login(
            body={
                "userId": self.userId,
                "password": self.password,
                "yob": self.yob
            }
        )

        self.session_token = json.loads(login)["sessionToken"]
        self.samco.set_session_token(self.session_token)

        print("Login successful")

    # ---------- WEBSOCKET CALLBACKS ----------

    def on_message(self, ws, message):
        
        message = json.loads(message)

        # #print msg
        # print("Stream ON MSG : ", message)

        # push tick to queue
        self.tick_queue.put(message)

    def on_error(self, ws, error):

        print("WebSocket Error:", error)

    def on_close(self, ws, code, msg):

        print("Connection Closed", code, msg)

    def on_open(self, ws):

        print("WebSocket connected")

        # correct payload like non-class working version
        symbols_json = json.dumps([{"symbol": self.symbol}])
        data = (
                '{"request":{"streaming_type":"quote", "data":{"symbols":'
                + symbols_json
                + '},"request_type":"subscribe", "response_format":"json"}}'
        )

        ws.send(data)
        ws.send("\n")

    # ---------- FLUSH WORKER ----------

    def write_files_per_5s(self):
        # print("HELLO1")
        batch = []
        last_flush = time.time()
        # symbol_folder = f"{self.output_path}/{self.symbol}"
        symbol_folder = f"{self.output_path}"
        while not self.stop_event.is_set():
            # print("HELLO2")
            try:
                msg = self.tick_queue.get(timeout=1)
                batch.append(msg)
            except Exception as e:
                # print("QUEUE ERRORRRRRRRRRRRRRRRRR: ", e)
                pass

            if time.time() - last_flush >= 5:
                # print("INSIDE")
                if batch:
                    # print("INSIDE2")
                    folder_minute = "minute_" + time.strftime("%Y%m%d_%H%M", time.localtime(time.time()))
                    # Create the directory if it doesn't exist
                    os.makedirs(f"{symbol_folder}/{folder_minute}", exist_ok=True)
                    file_name = f"{symbol_folder}/{folder_minute}/ticks_{int(time.time())}.json"
                    with open(file_name, "w") as f:
                        for msg in batch:
                            f.write(json.dumps(msg) + "\n")
                    # print("Flushed", len(batch), "ticks")
                    batch = []
                last_flush = time.time()

    # ---------- START STREAM ----------

    def start_stream(self, symbol):

        self.symbol = symbol

        headers = {"x-session-token": self.session_token}

        websocket.enableTrace(True)

        self.ws = websocket.WebSocketApp(
            "wss://stream.samco.in",
            header=headers,
            on_open=lambda ws: self.on_open(ws),
            on_message=lambda ws, msg: self.on_message(ws, msg),
            on_error=lambda ws, err: self.on_error(ws, err),
            on_close=lambda ws, code, msg: self.on_close(ws, code, msg)
        )

        # websocket thread
        ws_thread = threading.Thread(target=self.ws.run_forever)
        ws_thread.daemon = False
        ws_thread.start()
        
        # flush worker thread
        worker_thread = threading.Thread(target=self.write_files_per_5s)
        worker_thread.daemon = False
        worker_thread.start()
        
        # keep main thread alive
        ws_thread.join()
        worker_thread.join()

        print("Streaming started for", symbol)

    # ---------- STOP STREAM ----------

    def stop_stream(self):
        if self.ws:
            self.ws.close()
        print("Streaming stopped")

    # ---------- STOP WRITING FILES ----------

    def stop_writing_files(self):

        self.stop_event.set()
        print("writing files stopped")

In [0]:
stream = SamcoMarketStreamer(
    userId,
    password,
    yob,
    output_path
)

In [0]:
stream.start_stream(symbol)

In [0]:
# stream.stop_stream()

In [0]:
# stream.stop_writing_files()

In [0]:
# a = stream.tick_queue.get(timeout=1)
# print(a)

In [0]:
# stream.samco.get_quote(exchange=samco.EXCHANGE_MCX, symbol_name="GOLDM")

In [0]:
# stream.samco.get_quote(exchange=stream.samco.EXCHANGE_MCX, symbol_name="GOLDM")

In [0]:
# print(stream.samco.get_masters(exchange="MCX")
# )